# 04 — POMDP + belief state, visually

Now the world contains a fact you cannot directly see: **is the Sword actually active?**

Instead, you get a noisy clue: `glint` or `no glint`.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Partial observability</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">HIDDEN SWORD STATE</text><rect x="250.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="325.0" y="88" text-anchor="middle" font-size="15">NOISY SENSOR</text><rect x="475.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="550.0" y="88" text-anchor="middle" font-size="15">OBSERVATION</text><rect x="700.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="775.0" y="88" text-anchor="middle" font-size="15">BELIEF</text>
<line x1="180.0" y1="82" x2="242.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="405.0" y1="82" x2="467.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="630.0" y1="82" x2="692.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
from pathlib import Path
import sys, random, numpy as np, pandas as pd, matplotlib.pyplot as plt
repo = Path.cwd()
if repo.name == "notebooks": repo = repo.parent
sys.path.insert(0, str(repo))

P_GLINT_IF_ACTIVE = 0.85
P_GLINT_IF_DOWN = 0.10

plt.figure(figsize=(7,4))
plt.bar(["P(glint | active)","P(glint | down)"], [P_GLINT_IF_ACTIVE, P_GLINT_IF_DOWN])
plt.ylim(0,1)
plt.ylabel("Probability")
plt.title("Our noisy Sword sensor")
plt.show()

## 1. Observation is evidence, not truth

A `glint` makes Sword-active more likely, but does not prove it.

A belief state represents uncertainty over possible hidden states.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Belief update</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">PRIOR BELIEF</text><rect x="250.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="325.0" y="88" text-anchor="middle" font-size="15">OBSERVATION</text><rect x="475.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="550.0" y="88" text-anchor="middle" font-size="15">BAYES UPDATE</text><rect x="700.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="775.0" y="88" text-anchor="middle" font-size="15">POSTERIOR BELIEF</text>
<line x1="180.0" y1="82" x2="242.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="405.0" y1="82" x2="467.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="630.0" y1="82" x2="692.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
def bayes_update(prior_active, glint):
    if glint:
        la, ld = P_GLINT_IF_ACTIVE, P_GLINT_IF_DOWN
    else:
        la, ld = 1-P_GLINT_IF_ACTIVE, 1-P_GLINT_IF_DOWN
    num = la * prior_active
    return num / (num + ld*(1-prior_active))

prior = 0.5
post_glint = bayes_update(prior, True)
post_no = bayes_update(prior, False)

plt.figure(figsize=(8,4))
plt.bar(["prior","after glint","after no glint"], [prior, post_glint, post_no])
plt.ylim(0,1)
plt.ylabel("P(Sword active)")
plt.title("One observation changes belief")
plt.show()

## 2. History accumulates evidence

One noisy observation can be wrong.

A sequence of observations can make your belief more confident.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">History → belief</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">o₁</text><rect x="205.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="280.0" y="88" text-anchor="middle" font-size="15">o₂</text><rect x="385.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="460.0" y="88" text-anchor="middle" font-size="15">o₃</text><rect x="565.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="640.0" y="88" text-anchor="middle" font-size="15">o₄</text><rect x="745.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="820.0" y="88" text-anchor="middle" font-size="15">BELIEF AFTER HISTORY</text>
<line x1="180.0" y1="82" x2="197.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="360.0" y1="82" x2="377.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="540.0" y1="82" x2="557.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="720.0" y1="82" x2="737.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
rng = random.Random(4)
true_active = True
belief = 0.5
rows = []

for t in range(10):
    p = P_GLINT_IF_ACTIVE if true_active else P_GLINT_IF_DOWN
    glint = rng.random() < p
    belief = bayes_update(belief, glint)
    rows.append((t, glint, belief))

df = pd.DataFrame(rows, columns=["t","glint","P_active"])
display(df)

plt.figure(figsize=(9,4))
plt.plot(df["t"], df["P_active"], marker="o")
plt.axhline(1.0, linestyle="--", label="hidden truth = active")
plt.ylim(0,1)
plt.xlabel("Observation number")
plt.ylabel("Belief P(active)")
plt.title("Belief evolves as evidence arrives")
plt.legend()
plt.show()

## 3. Latest observation can be insufficient

Suppose the newest clue is `no glint`.

If the previous nine clues strongly favored Sword-active, your belief may still say active is very likely.

So `latest observation` and `state for decision-making` are not necessarily the same thing.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Why memory matters</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">LATEST OBS ONLY</text><rect x="325.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="400.0" y="88" text-anchor="middle" font-size="15">LOSE HISTORY</text><rect x="625.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="700.0" y="88" text-anchor="middle" font-size="15">AMBIGUOUS</text>
<line x1="180.0" y1="82" x2="317.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="480.0" y1="82" x2="617.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
histories = {
    "glint,glint,glint,no": [True,True,True,False],
    "no,no,no,no": [False,False,False,False],
    "glint,no,glint,no": [True,False,True,False],
}
finals = []

for name, hist in histories.items():
    b = 0.5
    for o in hist:
        b = bayes_update(b, o)
    finals.append(b)

plt.figure(figsize=(9,4))
plt.bar(histories.keys(), finals)
plt.ylim(0,1)
plt.ylabel("Final belief P(active)")
plt.xticks(rotation=15)
plt.title("Same latest observation ('no') — different beliefs because history differs")
plt.show()

## 4. A policy can act on belief

Instead of choosing from hidden truth, the policy chooses from its current belief.

Example:

- if `P(active) > 0.7`, avoid body attack
- otherwise, risk direct body damage


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Belief can be decision state</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">BELIEF P(active)</text><rect x="325.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="400.0" y="88" text-anchor="middle" font-size="15">POLICY THRESHOLD</text><rect x="625.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="700.0" y="88" text-anchor="middle" font-size="15">SAFE / RISKY ACTION</text>
<line x1="180.0" y1="82" x2="317.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="480.0" y1="82" x2="617.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
beliefs = np.linspace(0,1,101)
safe = (beliefs > 0.7).astype(int)

plt.figure(figsize=(9,4))
plt.plot(beliefs, safe)
plt.xlabel("Belief P(Sword active)")
plt.ylabel("Choose safe Sword-targeting move?")
plt.yticks([0,1], ["NO","YES"])
plt.title("Simple belief-based policy")
plt.show()

## 5. Map this to LifeOS

Hidden state can include facts you do not directly observe yet: someone's intent, an external approval, an unresolved dependency, or the user's current objective.

Telemetry gives evidence. History changes your internal estimate.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">LifeOS as partial observability</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">HIDDEN WORLD</text><rect x="250.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="325.0" y="88" text-anchor="middle" font-size="15">SCREEN / EMAIL / HISTORY</text><rect x="475.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="550.0" y="88" text-anchor="middle" font-size="15">BELIEF / INTERNAL STATE</text><rect x="700.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="775.0" y="88" text-anchor="middle" font-size="15">ACTION</text>
<line x1="180.0" y1="82" x2="242.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="405.0" y1="82" x2="467.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="630.0" y1="82" x2="692.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
sources = ["screen","calendar","messages","keyboard","history"]
uncertainty_reduction = [0.20,0.12,0.18,0.08,0.30]

plt.figure(figsize=(8,4))
plt.bar(sources, uncertainty_reduction)
plt.ylabel("Illustrative information contribution")
plt.title("Different observations can reduce uncertainty about hidden task state")
plt.show()

## Memorize

**Observation = what I saw. History = what I've seen/done. Belief = what I think the hidden world probably is given that history.**

Next: neural-network weights.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">POMDP mental model</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">OBSERVATION</text><rect x="250.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="325.0" y="88" text-anchor="middle" font-size="15">HISTORY</text><rect x="475.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="550.0" y="88" text-anchor="middle" font-size="15">BELIEF</text><rect x="700.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="775.0" y="88" text-anchor="middle" font-size="15">POLICY</text>
<line x1="180.0" y1="82" x2="242.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="405.0" y1="82" x2="467.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="630.0" y1="82" x2="692.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>
